# Chapter 1 · Foundations of Neural Networks
Practical Deep Learning with Python: Neural Networks, CNNs & Transfer Learning  
Nazmus Sayadat Rumman

Steps:
1. Download UCI_Credit_Card.csv (Default of Credit Card Clients, UCI, CC BY 4.0).
2. On Kaggle: Add Input → Upload the CSV. Session options → Accelerator → GPU.
3. Run All.

### D1-01 · Setup: imports, seed, versions

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
keras.utils.set_random_seed(42)
print("TF", tf.__version__, "| Keras", keras.__version__)

### D1-02 · One neuron: z = w·x + b, a = sigmoid(z)

In [ ]:
x = np.array([2.0, 3.0])
w = np.array([0.4, -0.2])
b = 0.1
z = w @ x + b
a = 1 / (1 + np.exp(-z))
print("z =", round(z, 4), "| a = sigmoid(z) =", round(a, 4))

### D1-03 · Perceptron = one straight line (AND gate)

In [ ]:
X_gate = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])
w_p = np.array([1.0, 1.0])
b_p = -1.5
pred = (X_gate @ w_p + b_p > 0).astype(int)
print("AND targets :", y_and)
print("Perceptron  :", pred)

### D1-03b · Plot the perceptron's line

In [ ]:
x1_line = np.linspace(-0.5, 1.5, 50)
plt.scatter(X_gate[:, 0], X_gate[:, 1], c=y_and, cmap="bwr", s=150, edgecolors="k")
plt.plot(x1_line, 1.5 - x1_line, "k--", label="x1 + x2 - 1.5 = 0")
plt.xlabel("x1"); plt.ylabel("x2"); plt.legend()
plt.title("Perceptron = one straight line")
plt.show()

### D1-04 · XOR: one line fails, a hidden layer solves it

In [ ]:
y_xor = np.array([0, 1, 1, 0])
step = lambda v: (v > 0).astype(int)
h1 = step(X_gate @ np.array([1, 1]) - 0.5)   # hidden neuron 1 = OR
h2 = step(X_gate @ np.array([1, 1]) - 1.5)   # hidden neuron 2 = AND
out = step(h1 - h2 - 0.5)                    # output = OR but not AND
print("XOR targets   :", y_xor)
print("Hidden h1 (OR):", h1, "| h2 (AND):", h2)
print("Network output:", out)

### D1-05 · MLP anatomy: the 2-3-2 network's weights and biases

In [ ]:
W1 = np.array([[0.5, -0.3, 0.8],
               [0.2,  0.4, -0.6]])
b1 = np.array([0.1, -0.1, 0.2])
W2 = np.array([[ 0.7, -0.5],
               [-0.2,  0.6],
               [ 0.3,  0.9]])
b2 = np.array([0.0, 0.1])
print("W1", W1.shape, "b1", b1.shape, "W2", W2.shape, "b2", b2.shape)
print("Total parameters:", W1.size + b1.size + W2.size + b2.size)

### D1-06 · Activations: ReLU, sigmoid, tanh

In [ ]:
z_vals = np.array([-2.0, 0.0, 2.0])
print("z      :", z_vals)
print("ReLU   :", np.maximum(0, z_vals))
print("sigmoid:", (1 / (1 + np.exp(-z_vals))).round(4))
print("tanh   :", np.tanh(z_vals).round(4))

### D1-06b · Plot the three activations

In [ ]:
zz = np.linspace(-4, 4, 200)
plt.plot(zz, np.maximum(0, zz), label="ReLU")
plt.plot(zz, 1 / (1 + np.exp(-zz)), label="sigmoid")
plt.plot(zz, np.tanh(zz), label="tanh")
plt.axhline(0, color="gray", lw=0.5); plt.legend()
plt.title("Activation functions")
plt.show()

### D1-07 · Softmax: logits → probabilities

In [ ]:
def softmax(z):
    e = np.exp(z - np.max(z))
    return e / e.sum()

logits = np.array([2.0, 1.0, 0.1])
p = softmax(logits)
print("softmax       :", p.round(4), "| sum =", p.sum().round(4))
print("logits + 1000 :", softmax(logits + 1000).round(4))

### D1-08 · Two-class softmax = sigmoid of the difference

In [ ]:
z_two = np.array([1.5, 0.3])
print("softmax, class 1 :", softmax(z_two)[0].round(4))
print("sigmoid(z1 - z2) :", (1 / (1 + np.exp(-(z_two[0] - z_two[1])))).round(4))

### D1-09 · Forward pass, hidden layer

In [ ]:
x = np.array([1.0, 2.0])
z1 = x @ W1 + b1
a1 = np.maximum(0, z1)
print("z1 =", z1.round(4))
print("a1 = ReLU(z1) =", a1.round(4))

### D1-10 · Forward pass, output layer + softmax

In [ ]:
z2 = a1 @ W2 + b2
p_numpy = softmax(z2)
print("z2 (logits)     =", z2.round(4))
print("p = softmax(z2) =", p_numpy.round(4))

### D1-11 · The same 2-3-2 network in Keras

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(2,)),
    keras.layers.Dense(3, activation="relu"),
    keras.layers.Dense(2, activation="softmax"),
])
model.summary()

### D1-12 · Load our hand weights into Keras and compare

In [ ]:
model.set_weights([W1, b1, W2, b2])
p_keras = model.predict(x.reshape(1, 2), verbose=0)[0]
print("NumPy :", p_numpy.round(4))
print("Keras :", p_keras.round(4))
print("Match :", np.allclose(p_numpy, p_keras, atol=1e-6))

### D1-13 · Kaggle: is the GPU attached?

In [ ]:
print(tf.config.list_physical_devices("GPU"))

### D1-14 · Load the credit card dataset

In [ ]:
import glob
import pandas as pd
path = glob.glob("/kaggle/input/**/UCI_Credit_Card.csv", recursive=True) + glob.glob("UCI_Credit_Card.csv")
df = pd.read_csv(path[0])
print(df.shape)
df.head()

### D1-15 · Target balance: how many clients defaulted?

In [ ]:
counts = df["default.payment.next.month"].value_counts()
print(counts)
print("Default rate:", round(counts[1] / len(df), 4))

### D1-16 · An untrained network on 5 real clients

In [ ]:
X = df.drop(columns=["ID", "default.payment.next.month"])
X5 = ((X - X.mean()) / X.std()).head(5).to_numpy()
credit_net = keras.Sequential([
    keras.Input(shape=(23,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(2, activation="softmax"),
])
print("Parameters:", credit_net.count_params())
print(credit_net.predict(X5, verbose=0).round(3))